# Synthetic vs original: what changed, and does the real data help? (S6E11 fraud)

This month's data was generated from the **Credit Card Fraud Detection 2026** dataset (20k rows, readable columns).
Playground data is "close to, but not exactly" the original, so three questions matter before any heavy modelling:

1. **What drifted?** Which features look different in the competition data vs the original?
2. **Is the fraud signal the same?** Does each feature separate fraud in both datasets in the same way?
3. **Does adding the original rows help?** A LightGBM baseline with and without them, same folds, same seed.

Everything runs on CPU in a few minutes. Metric: ROC AUC.


In [ ]:
# all imports in one place
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)
SEED = 42

# competition files
COMP = os.path.dirname(next(iter(glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True))))
train = pd.read_csv(f'{COMP}/train.csv')
test = pd.read_csv(f'{COMP}/test.csv')
sub = pd.read_csv(f'{COMP}/sample_submission.csv')

# original dataset: any csv in the inputs that is not a competition file
orig_path = [p for p in glob.glob('/kaggle/input/**/*.csv', recursive=True) if os.path.dirname(p) != COMP][0]
orig = pd.read_csv(orig_path)

target = 'is_fraud'
if target not in orig:   # fall back to the original's fraud label column
    target_orig = [c for c in orig.columns if 'fraud' in c.lower() and 'ai' not in c.lower()][0]
    orig = orig.rename(columns={target_orig: target})

print('train', train.shape, '| test', test.shape, '| original', orig.shape, '|', os.path.basename(orig_path))
print(f'fraud rate: train {train[target].mean():.4f} | original {orig[target].mean():.4f}')


In [ ]:
# features shared by all three tables (drops ids / columns that only exist in one place)
features = [c for c in train.columns if c not in ('id', target) and c in test.columns and c in orig.columns]
cat_cols = [c for c in features if not pd.api.types.is_numeric_dtype(train[c]) or train[c].dtype == 'bool']
num_cols = [c for c in features if c not in cat_cols]
# make the original's numeric columns numeric too (e.g. True/False flags stored as text)
for c in num_cols:
    orig[c] = pd.to_numeric(orig[c].replace({True: 1, False: 0, 'True': 1, 'False': 0}), errors='coerce')
only_orig = [c for c in orig.columns if c not in train.columns]
print(len(features), 'shared features |', len(cat_cols), 'categorical |', len(num_cols), 'numeric')
print('columns only in the original (not used):', only_orig)
train[features].head()


## 1. What drifted?

For numeric features we use the **KS statistic** (0 = same distribution, 1 = completely different).
For categorical features, the **total variation distance** between category shares (same 0-1 scale).


In [ ]:
def tv_distance(a, b):
    """Total variation distance between two categorical distributions."""
    pa, pb = a.value_counts(normalize=True), b.value_counts(normalize=True)
    cats = pa.index.union(pb.index)
    return 0.5 * np.abs(pa.reindex(cats, fill_value=0) - pb.reindex(cats, fill_value=0)).sum()

rows = []
for c in features:
    if c in num_cols:
        d_orig = ks_2samp(train[c], orig[c]).statistic
        d_test = ks_2samp(train[c], test[c]).statistic
        kind = 'KS'
    else:
        d_orig = tv_distance(train[c].astype(str), orig[c].astype(str))
        d_test = tv_distance(train[c].astype(str), test[c].astype(str))
        kind = 'TV'
    rows.append({'feature': c, 'type': kind, 'train_vs_original': d_orig, 'train_vs_test': d_test})
drift = pd.DataFrame(rows).sort_values('train_vs_original', ascending=False).reset_index(drop=True)
print(drift.round(3).to_string(index=False))


In [ ]:
# the 6 most drifted numeric features: competition train vs original
top = [c for c in drift.feature if c in num_cols][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, c in zip(axes.ravel(), top):
    lo, hi = np.percentile(pd.concat([train[c], orig[c]]), [0.5, 99.5])
    bins = np.linspace(lo, hi, 50) if hi > lo else 20
    ax.hist(train[c], bins=bins, density=True, alpha=.55, label='competition train', color='#1f4e79')
    ax.hist(orig[c], bins=bins, density=True, alpha=.55, label='original', color='#e07b39')
    ax.set_title(c)
axes[0, 0].legend()
plt.suptitle('Most drifted numeric features', y=1.01); plt.tight_layout(); plt.show()


## 2. Can a model tell them apart? (adversarial validation)

Train LightGBM to predict "which table is this row from".
AUC near **0.5** = indistinguishable; near **1.0** = easy to tell apart.
Train vs test should be ~0.5 (same generator). Train vs original tells us how far the generator drifted.


In [ ]:
def to_lgb(df):
    """Categoricals as pandas 'category' with one shared set of levels."""
    out = df[features].copy()
    for c in cat_cols:
        out[c] = pd.Categorical(out[c].astype(str), categories=levels[c])
    return out

levels = {c: sorted(set(train[c].astype(str)) | set(test[c].astype(str)) | set(orig[c].astype(str))) for c in cat_cols}
params_adv = dict(n_estimators=300, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.8, random_state=SEED, verbose=-1)

def adversarial_auc(a, b, n=100_000):
    """CV AUC of a model separating table a from table b (sampled to keep it fast)."""
    a = a.sample(min(n, len(a)), random_state=SEED); b = b.sample(min(n, len(b)), random_state=SEED)
    X = to_lgb(pd.concat([a, b], ignore_index=True))
    y = np.r_[np.zeros(len(a)), np.ones(len(b))]
    oof = np.zeros(len(y)); imp = pd.Series(0.0, index=features)
    for tr, va in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y):
        m = lgb.LGBMClassifier(**params_adv).fit(X.iloc[tr], y[tr])
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
        imp += pd.Series(m.booster_.feature_importance('gain'), index=features)
    return roc_auc_score(y, oof), imp.sort_values(ascending=False)

auc_tt, _ = adversarial_auc(train, test)
auc_to, imp_to = adversarial_auc(train, orig)
print(f'adversarial AUC  train vs test: {auc_tt:.3f} | train vs original: {auc_to:.3f}')
print('features that give the original away:'); print((imp_to / imp_to.sum()).head(8).round(3))


## 3. Is the fraud signal the same?

For each feature: how well does it separate fraud on its own (**univariate AUC**, folded so 0.5 = useless, 1 = perfect),
in the competition data vs the original. Points on the diagonal = same signal in both.


In [ ]:
def single_auc(x, y):
    """AUC of one feature used alone as a score (categoricals scored by their fraud rate)."""
    if not pd.api.types.is_numeric_dtype(x) or x.dtype == 'bool':
        x = x.astype(str).map(pd.Series(y.values, index=x.index).groupby(x.astype(str)).mean())
    a = roc_auc_score(y, x.fillna(x.median()))
    return max(a, 1 - a)

sig = pd.DataFrame({'competition': [single_auc(train[c], train[target]) for c in features],
                    'original': [single_auc(orig[c], orig[target]) for c in features]}, index=features)
sig = sig.sort_values('competition', ascending=False)
print(sig.round(3))

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(sig.original, sig.competition, color='#1f4e79')
for f, r in sig.head(8).iterrows():
    ax.annotate(f, (r.original, r.competition), fontsize=8, xytext=(3, 3), textcoords='offset points')
ax.plot([.5, 1], [.5, 1], '--', color='grey'); ax.set_xlim(.48, 1); ax.set_ylim(.48, 1)
ax.set_xlabel('univariate AUC, original'); ax.set_ylabel('univariate AUC, competition')
ax.set_title('Fraud signal per feature'); plt.tight_layout(); plt.show()


## 4. Does adding the original rows help?

Same 5 folds, same LightGBM, two settings:
* **A** - competition train only
* **B** - competition train + all original rows added to every training fold (never to validation)

Validation is always on competition rows only, so the two CV scores are directly comparable.


In [ ]:
X = to_lgb(train); y = train[target].values
X_orig = to_lgb(orig); y_orig = orig[target].values
X_te = to_lgb(test)

params = dict(n_estimators=3000, learning_rate=0.03, num_leaves=63, min_child_samples=50,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=1.0,
              random_state=SEED, verbose=-1)
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

def run(add_original):
    oof = np.zeros(len(y)); pred = np.zeros(len(X_te))
    for k, (tr, va) in enumerate(cv.split(X, y)):
        X_tr, y_tr = X.iloc[tr], y[tr]
        if add_original:
            X_tr = pd.concat([X_tr, X_orig], ignore_index=True); y_tr = np.r_[y_tr, y_orig]
        m = lgb.LGBMClassifier(**params).fit(X_tr, y_tr, eval_set=[(X.iloc[va], y[va])], eval_metric='auc',
                                             callbacks=[lgb.early_stopping(200, verbose=False)])
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
        pred += m.predict_proba(X_te)[:, 1] / cv.n_splits
        print(f'  fold {k}: AUC {roc_auc_score(y[va], oof[va]):.5f} | trees {m.best_iteration_}')
    return roc_auc_score(y, oof), oof, pred

results = {}
for name, flag in [('A: competition only', False), ('B: + original rows', True)]:
    print(name)
    results[name] = run(flag)
    print(f'  CV AUC {results[name][0]:.5f}')


In [ ]:
summary = pd.Series({k: v[0] for k, v in results.items()}, name='CV AUC')
print(summary.round(5))
best = summary.idxmax()
print(f'\nbest: {best} | gain from original rows: {summary.iloc[1] - summary.iloc[0]:+.5f} AUC')

sub[target] = results[best][2]
sub.to_csv('submission.csv', index=False)
sub.head()


## 5. Takeaways (numbers from this run)

* **The generator moved the money columns a lot.** `amount_usd` (KS 0.38), `account_balance_usd` (0.19) and
  `distance_from_home_km` (0.16) drifted most. A model tells original rows from competition rows with **AUC 0.90**,
  while train vs test is **0.50** (same generator, no shift to worry about between train and test).
* **The fraud signal mostly survived.** The top signals are the same in both tables: `merchant_risk_score`,
  `velocity_score`, `cvv_retry_count`, `auth_method`. Two differences: `customer_age` is a much stronger signal in the
  competition data (0.63 vs 0.54), and `time_of_day_hour` carries signal in the original (0.57) but **none** here (0.50).
* **Adding the 20k original rows did not help:** CV AUC 0.94619 without vs 0.94604 with (-0.00016, same folds).
  With 700k generated rows and a 0.90 adversarial AUC, the original is too different and too small to add information.
  It may still help as a **feature source** (e.g. fraud rate per category from the original) rather than as extra rows.
* **Baseline to beat: 0.9462 CV AUC** with a plain LightGBM, no feature engineering.
* Next ideas: count/frequency encodings of the numeric columns (common in generated Playground data), CatBoost on the
  11 categoricals, target encoding inside folds, and blending.

*Data: competition data (Kaggle Playground S6E11) and the original Credit Card Fraud Detection 2026 dataset (CC0) by Udit Jain.*
